# MEAQ-T Advanced: Pareto Optimization & Constraint Filtering

This notebook demonstrates advanced optimization techniques including:
- Pareto front extraction
- Hard constraint filtering
- Weight preset comparison
- Multi-scenario analysis

**Duration**: ~10 minutes

## Setup

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from meaqt import optimize_stacks, pareto_front, save_pareto_scatter

print('✓ Imports successful')

## Scenario 1: Balanced Weight Preset

In [ ]:
balanced = optimize_stacks(preset='balanced', top_k=15, score_mode='raw')
print(f"Balanced optimization: {len(balanced)} stacks")
print(balanced[['protective', 'interlayer', 'score']].head(5))

## Scenario 2: Extreme Ultra-High-Temperature (UHT)

In [ ]:
uht = optimize_stacks(preset='extreme_uht', top_k=15, score_mode='raw')
print(f"UHT optimization: {len(uht)} stacks")
print(uht[['protective', 'interlayer', 'score']].head(5))

# Compare top choices
print("\n=== Comparison ===\n")
print("Balanced top:")
print(f"  Heat tol: {balanced.iloc[0]['heat_tol']:.3f}, Radiation tol: {balanced.iloc[0]['radiation_tol']:.3f}")
print("\nUHT top:")
print(f"  Heat tol: {uht.iloc[0]['heat_tol']:.3f}, Radiation tol: {uht.iloc[0]['radiation_tol']:.3f}")

## Scenario 3: Hard Constraints

Filter stacks by maximum acceptable thermal conductivity and minimum coupling.

In [ ]:
constrained = optimize_stacks(
    preset='balanced',
    top_k=10,
    max_thermal_kappa_wmk=2.0,  # Low-kappa constraint
    min_coupling=0.75,            # Strong coupling required
    score_mode='raw'
)

print(f"Constrained results: {len(constrained)} stacks meet criteria")
if len(constrained) > 0:
    print("\nTop constrained stack:")
    top = constrained.iloc[0]
    print(f"  Thermal kappa: {top['thermal_kappa_wmk']:.3f} W/m-K")
    print(f"  Coupling: {top['coupling']:.3f}")
    print(f"  Score: {top['score']:.4f}")
else:
    print("No stacks meet the constraints.")

## Scenario 4: Pareto Front Extraction

Identify non-dominated solutions (trade-offs between competing objectives).

In [ ]:
# Get all stacks first
all_stacks = optimize_stacks(preset='balanced', top_k=80, score_mode='normalized')

# Extract Pareto-optimal solutions
pareto = pareto_front(
    all_stacks,
    maximize=['heat_tol', 'radiation_tol', 'healing_rate', 'coupling'],
    minimize=['thermal_kappa_wmk']
)

print(f"Pareto front size: {len(pareto)} out of {len(all_stacks)} candidates")
print("\nPareto-optimal stacks:")
for i, row in pareto.iterrows():
    print(f"  [{i+1}] {row['protective']:<25} | "
          f"Kappa: {row['thermal_kappa_wmk']:5.2f} | "
          f"Heal: {row['healing_rate']:5.2f} | "
          f"Score: {row['score']:6.4f}")

## Visualization: Pareto Trade-offs

In [ ]:
# Plot all vs. Pareto-optimal
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# All stacks
ax = axes[0]
ax.scatter(all_stacks['thermal_kappa_wmk'], all_stacks['healing_rate'], 
           alpha=0.4, s=50, c='lightblue', edgecolors='blue', label='All')
ax.scatter(pareto['thermal_kappa_wmk'], pareto['healing_rate'], 
           alpha=0.8, s=100, c='red', edgecolors='darkred', label='Pareto')
ax.set_xlabel('Thermal Conductivity (W/m-K)')
ax.set_ylabel('Healing Rate')
ax.set_title('Pareto Trade-off: Thermal Conductivity vs. Healing')
ax.legend()
ax.grid(alpha=0.3)

# Pareto solutions colored by score
ax = axes[1]
sc = ax.scatter(pareto['thermal_kappa_wmk'], pareto['healing_rate'],
                c=pareto['score'], s=150, cmap='viridis', edgecolors='black', linewidth=1)
ax.set_xlabel('Thermal Conductivity (W/m-K)')
ax.set_ylabel('Healing Rate')
ax.set_title('Pareto Front: Solutions Colored by Score')
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Score')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('pareto_analysis.png', dpi=150, bbox_inches='tight')
print("✓ Saved pareto_analysis.png")
plt.show()

## Summary & Key Takeaways

1. **Weight presets** allow targeting different regimes (balanced, low-temp prototype, UHT).
2. **Hard constraints** ensure solutions meet critical requirements.
3. **Pareto front** reveals trade-offs without arbitrary ranking.
4. **Normalized scores** enable fair multi-scale comparisons.